# 01 — Common Data Quality Audit

**ผู้รับผิดชอบ:** ทอฝัน  
**วันที่ตรวจ:** 18 สิงหาคม 2026  
**ขอบเขต:** ตรวจข้อมูลที่ผ่านการทำความสะอาดแล้วทั้ง 5 ตารางใน `data/processed/` โดยไม่แก้ไขไฟล์ต้นฉบับใน `data/raw/`

Notebook นี้ตรวจ schema, ค่าว่าง, แถวซ้ำ, primary key, foreign key, ความสอดคล้องข้ามตาราง, ช่วงค่า, ตรรกะด้านเวลา และเปรียบเทียบคุณภาพข้อมูลก่อน/หลังทำความสะอาด

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from kaverentai.data.load_data import TABLES, load_all_data
from kaverentai.data.validate_data import validate_tables

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

tables = load_all_data(cleaned=True, data_dir=PROCESSED_DIR)
raw_tables = load_all_data(cleaned=False, data_dir=RAW_DIR)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

print(f'Project root: {PROJECT_ROOT}')
print(f'Loaded tables: {list(tables)}')

Project root: D:\final project
Loaded tables: ['projects', 'units', 'listings', 'leases', 'weekly_market']


## 1. ภาพรวมและ schema ขั้นพื้นฐาน

ทุกตารางต้องโหลดได้ ไม่เป็นตารางว่าง และมีคอลัมน์บังคับตาม `validate_data.py`

In [2]:
schema_errors = validate_tables(tables)
inventory = pd.DataFrame([
    {
        'table': name,
        'rows': len(frame),
        'columns': len(frame.columns),
        'duplicate_rows': int(frame.duplicated().sum()),
        'memory_mb': frame.memory_usage(deep=True).sum() / 1024**2,
    }
    for name, frame in tables.items()
]).set_index('table')

display(inventory.style.format({'rows': '{:,}', 'memory_mb': '{:.2f}'}))
print('Schema validation:', 'PASS' if not schema_errors else 'FAIL')
if schema_errors:
    display(schema_errors)

,rows,columns,duplicate_rows,memory_mb
table,,,,
projects,19,7,0,0.00
units,"15,367",8,0,1.17
listings,"31,544",24,0,8.18
leases,"50,965",10,0,5.03
weekly_market,"2,288",9,0,0.19


Schema validation: PASS


## 2. Data dictionary

ตารางด้านล่างรวมชนิดข้อมูล จำนวนค่าที่ไม่ว่าง จำนวนค่าที่ไม่ซ้ำ และความหมายของแต่ละคอลัมน์จากข้อมูลที่โหลดจริง

In [3]:
COLUMN_DESCRIPTIONS = {
    'project_id': 'รหัสโครงการ ใช้เชื่อมตาราง projects',
    'project_name': 'ชื่อโครงการ',
    'university': 'สถานศึกษาหรือทำเลหลักของโครงการ',
    'distance_to_campus_m': 'ระยะทางถึงสถานศึกษา (เมตร)',
    'total_units': 'จำนวนห้องทั้งหมดของโครงการ',
    'year_completed': 'ปีที่โครงการสร้างเสร็จ',
    'facility_count': 'จำนวนกิจกรรม/สิ่งอำนวยความสะดวกส่วนกลาง',
    'unit_id': 'รหัสห้อง ใช้เชื่อมตาราง units',
    'floor': 'ชั้นของห้อง',
    'size_sqm': 'ขนาดห้อง (ตารางเมตร)',
    'room_type': 'ประเภทห้อง: studio, 1BR หรือ 1BR Plus',
    'view': 'ประเภทวิว: standard, city หรือ pool',
    'furnished': 'สถานะเฟอร์นิเจอร์ครบ',
    'agent_id': 'รหัสนายหน้าที่ดูแลห้อง/ประกาศ',
    'listing_id': 'รหัสประกาศเช่า',
    'week_listed': 'ลำดับสัปดาห์ที่ลงประกาศ เริ่มจาก 0',
    'date_listed': 'วันที่ลงประกาศ รูปแบบ YYYY-MM-DD',
    'season_listed': 'ฤดูกาล ณ วันที่ลงประกาศ',
    'asking_rent': 'ค่าเช่าล่าสุดที่ประกาศ (บาท/เดือน)',
    'first_asking_rent': 'ค่าเช่าที่ตั้งครั้งแรก (บาท/เดือน)',
    'weeks_on_market': 'จำนวนสัปดาห์ที่ประกาศอยู่ในตลาด',
    'n_viewings': 'จำนวนครั้งที่มีผู้เข้าชมห้อง',
    'leased': 'สถานะว่าประกาศปล่อยเช่าได้แล้ว',
    'week_leased': 'ลำดับสัปดาห์ที่ปล่อยเช่าได้; ว่างเมื่อยังไม่ถูกเช่า',
    'tenant_segment': 'กลุ่มผู้เช่า; ว่างเมื่อยังไม่ถูกเช่า',
    'lease_months': 'ระยะสัญญาเช่า (เดือน)',
    'lease_id': 'รหัสสัญญาเช่า',
    'week_start': 'ลำดับสัปดาห์ที่เริ่มสัญญา',
    'date_start': 'วันที่เริ่มสัญญา รูปแบบ YYYY-MM-DD',
    'week_end': 'ลำดับสัปดาห์ที่สิ้นสุดสัญญา',
    'rent': 'ค่าเช่าตามสัญญา (บาท/เดือน)',
    'is_renewal': 'ระบุว่าสัญญานี้เป็นการต่อสัญญา',
    'week': 'ลำดับสัปดาห์ เริ่มจาก 0',
    'date': 'วันที่เริ่มสัปดาห์ รูปแบบ YYYY-MM-DD',
    'season': 'ฤดูกาลของสัปดาห์',
    'n_searchers': 'จำนวนผู้ค้นหาห้องในสัปดาห์นั้น',
    'n_listings_open': 'จำนวนประกาศที่ยังเปิดอยู่',
    'n_units_leased': 'จำนวนห้องที่มีผู้เช่าอยู่',
    'occupancy': 'สัดส่วนห้องที่มีผู้เช่า ช่วง 0 ถึง 1',
    'median_asking_rent': 'ค่ามัธยฐานของค่าเช่าที่ประกาศ; ว่างได้เมื่อไม่มีประกาศเปิด',
}

dictionary_rows = []
for table_name, frame in tables.items():
    for column in frame.columns:
        dictionary_rows.append({
            'table': table_name,
            'column': column,
            'dtype': str(frame[column].dtype),
            'non_null': int(frame[column].notna().sum()),
            'null': int(frame[column].isna().sum()),
            'unique': int(frame[column].nunique(dropna=True)),
            'description': COLUMN_DESCRIPTIONS.get(column, ''),
        })

data_dictionary = pd.DataFrame(dictionary_rows)
display(data_dictionary.style.format({'non_null': '{:,}', 'null': '{:,}', 'unique': '{:,}'}))

,table,column,dtype,non_null,null,unique,description
0,projects,project_id,int64,19,0,19,รหัสโครงการ ใช้เชื่อมตาราง projects
1,projects,project_name,str,19,0,19,ชื่อโครงการ
2,projects,university,str,19,0,11,สถานศึกษาหรือทำเลหลักของโครงการ
3,projects,distance_to_campus_m,int64,19,0,12,ระยะทางถึงสถานศึกษา (เมตร)
4,projects,total_units,int64,19,0,18,จำนวนห้องทั้งหมดของโครงการ
5,projects,year_completed,int64,19,0,9,ปีที่โครงการสร้างเสร็จ
6,projects,facility_count,int64,19,0,9,จำนวนกิจกรรม/สิ่งอำนวยความสะดวกส่วนกลาง
7,units,unit_id,str,"15,367",0,"15,367",รหัสห้อง ใช้เชื่อมตาราง units
8,units,project_id,int64,"15,367",0,19,รหัสโครงการ ใช้เชื่อมตาราง projects
9,units,floor,int64,"15,367",0,8,ชั้นของห้อง


## 3. ค่าว่าง

ค่าว่างไม่ได้เป็นข้อผิดพลาดเสมอไป จึงตรวจร่วมกับเงื่อนไขทางธุรกิจ:

- `week_leased`, `tenant_segment`, `lease_months` ต้องว่างเฉพาะประกาศที่ `leased=False`
- `median_asking_rent` ว่างได้เฉพาะสัปดาห์ที่ `n_listings_open=0`

In [4]:
missing_rows = []
for table_name, frame in tables.items():
    for column, count in frame.isna().sum().items():
        if count:
            missing_rows.append({
                'table': table_name,
                'column': column,
                'missing': int(count),
                'missing_pct': count / len(frame) * 100,
            })
missing_summary = pd.DataFrame(missing_rows)
display(missing_summary.style.format({'missing': '{:,}', 'missing_pct': '{:.2f}%'}))

listings = tables['listings']
outcome_columns = ['week_leased', 'tenant_segment', 'lease_months']
leased_missing = int(listings.loc[listings['leased'], outcome_columns].isna().any(axis=1).sum())
unleased_has_values = int(listings.loc[~listings['leased'], outcome_columns].notna().any(axis=1).sum())

weekly = tables['weekly_market']
median_missing_with_open_listings = int(
    weekly.loc[weekly['median_asking_rent'].isna(), 'n_listings_open'].gt(0).sum()
)

missing_checks = pd.DataFrame([
    {'check': 'ประกาศที่เช่าแล้วต้องมีข้อมูลผลลัพธ์ครบ', 'failures': leased_missing},
    {'check': 'ประกาศที่ยังไม่เช่าต้องไม่มีข้อมูลผลลัพธ์', 'failures': unleased_has_values},
    {'check': 'median_asking_rent ว่างได้เมื่อไม่มีประกาศเปิดเท่านั้น', 'failures': median_missing_with_open_listings},
])
missing_checks['status'] = np.where(missing_checks['failures'].eq(0), 'PASS', 'FAIL')
display(missing_checks)

,table,column,missing,missing_pct
0,listings,week_leased,566,1.79%
1,listings,tenant_segment,566,1.79%
2,listings,lease_months,566,1.79%
3,weekly_market,median_asking_rent,3,0.13%


,check,failures,status
0,ประกาศที่เช่าแล้วต้องมีข้อมูลผลลัพธ์ครบ,0,PASS
1,ประกาศที่ยังไม่เช่าต้องไม่มีข้อมูลผลลัพธ์,0,PASS
2,median_asking_rent ว่างได้เมื่อไม่มีประกาศเปิดเท่านั้น,0,PASS


## 4. Primary key, foreign key และความสอดคล้องข้ามตาราง

In [5]:
quality_checks = []

def add_check(area, check, failures, note='', level='FAIL'):
    if not np.isscalar(failures):
        failures = np.asarray(failures).sum()
    failures = int(failures)
    quality_checks.append({
        'area': area,
        'check': check,
        'failures': failures,
        'status': 'PASS' if failures == 0 else level,
        'note': note,
    })

primary_keys = {
    'projects': ['project_id'],
    'units': ['unit_id'],
    'listings': ['listing_id'],
    'leases': ['lease_id'],
    'weekly_market': ['project_id', 'week'],
}
for table_name, key_columns in primary_keys.items():
    failures = tables[table_name].duplicated(key_columns).sum()
    add_check('Primary key', f"{table_name}: {', '.join(key_columns)} ไม่ซ้ำ", failures)

projects = tables['projects']
units = tables['units']
leases = tables['leases']
project_ids = set(projects['project_id'])
unit_ids = set(units['unit_id'])

foreign_keys = [
    ('units.project_id → projects', ~units['project_id'].isin(project_ids)),
    ('listings.project_id → projects', ~listings['project_id'].isin(project_ids)),
    ('listings.unit_id → units', ~listings['unit_id'].isin(unit_ids)),
    ('leases.project_id → projects', ~leases['project_id'].isin(project_ids)),
    ('leases.unit_id → units', ~leases['unit_id'].isin(unit_ids)),
    ('weekly_market.project_id → projects', ~weekly['project_id'].isin(project_ids)),
]
for label, invalid_mask in foreign_keys:
    add_check('Foreign key', label, invalid_mask.sum())

unit_project = units.set_index('unit_id')['project_id']
add_check(
    'Cross-table', 'listings.project_id ตรงกับ units.project_id',
    listings['project_id'].ne(listings['unit_id'].map(unit_project)).sum(),
)
add_check(
    'Cross-table', 'leases.project_id ตรงกับ units.project_id',
    leases['project_id'].ne(leases['unit_id'].map(unit_project)).sum(),
)

project_lookup = projects.set_index('project_id')
for column in ['project_name', 'university', 'distance_to_campus_m', 'facility_count']:
    add_check(
        'Cross-table', f'listings.{column} ตรงกับ projects.{column}',
        listings[column].ne(listings['project_id'].map(project_lookup[column])).sum(),
    )

unit_lookup = units.set_index('unit_id')
for column in ['floor', 'size_sqm', 'room_type', 'view', 'furnished']:
    add_check(
        'Cross-table', f'listings.{column} ตรงกับ units.{column}',
        listings[column].ne(listings['unit_id'].map(unit_lookup[column])).sum(),
    )

display(pd.DataFrame(quality_checks))

,area,check,failures,status,note
0,Primary key,projects: project_id ไม่ซ้ำ,0,PASS,
1,Primary key,units: unit_id ไม่ซ้ำ,0,PASS,
2,Primary key,listings: listing_id ไม่ซ้ำ,0,PASS,
3,Primary key,leases: lease_id ไม่ซ้ำ,0,PASS,
4,Primary key,"weekly_market: project_id, week ไม่ซ้ำ",0,PASS,
5,Foreign key,units.project_id → projects,0,PASS,
6,Foreign key,listings.project_id → projects,0,PASS,
7,Foreign key,listings.unit_id → units,0,PASS,
8,Foreign key,leases.project_id → projects,0,PASS,
9,Foreign key,leases.unit_id → units,0,PASS,


## 5. ช่วงค่า หมวดหมู่ และตรรกะภายในตาราง

เงื่อนไขถูกเลือกจากความหมายของข้อมูล ไม่ได้ใช้ IQR ตัดค่าที่ดูสุดโต่งออกโดยอัตโนมัติ เพราะค่าจริงบางค่า เช่นระยะห่างมหาวิทยาลัย 4,600 เมตร อาจเป็นค่าถูกต้อง

In [6]:
# Units
add_check('Domain', 'units.floor อยู่ในช่วง 1–8', ~units['floor'].between(1, 8))
add_check('Domain', 'units.size_sqm อยู่ในช่วงสมเหตุผล 20–50', ~units['size_sqm'].between(20, 50))
add_check('Domain', 'units.room_type อยู่ในหมวดที่กำหนด', ~units['room_type'].isin(['studio', '1BR', '1BR Plus']))
add_check('Domain', 'units.view อยู่ในหมวดที่กำหนด', ~units['view'].isin(['standard', 'city', 'pool']))

# Listings
add_check('Domain', 'ค่าเช่าใน listings เป็นบวก', (listings[['asking_rent', 'first_asking_rent']] <= 0).any(axis=1))
add_check('Domain', 'weeks_on_market และ n_viewings ไม่ติดลบ', (listings[['weeks_on_market', 'n_viewings']] < 0).any(axis=1))
add_check('Domain', 'season_listed อยู่ในหมวดที่กำหนด', ~listings['season_listed'].isin(['normal', 'peak', 'shoulder', 'off']))
add_check('Logic', 'week_leased = week_listed + weeks_on_market เมื่อเช่าแล้ว',
          listings.loc[listings['leased'], 'week_leased'].ne(
              listings.loc[listings['leased'], 'week_listed'] + listings.loc[listings['leased'], 'weeks_on_market']
          ).sum())
rent_increases = listings['asking_rent'].gt(listings['first_asking_rent'])
increase_note = (
    f"พบ {rent_increases.sum():,} รายการ; ทั้งหมดเป็น listing_id ที่ลงท้าย -B "
    f"และเพิ่มสูงสุด {(listings.loc[rent_increases, 'asking_rent'] / listings.loc[rent_increases, 'first_asking_rent'] - 1).max() * 100:.2f}% "
    'ควรยืนยันนิยาม first_asking_rent ก่อนนำไปสร้าง feature'
)
add_check('Logic', 'asking_rent ไม่สูงกว่า first_asking_rent', rent_increases.sum(), increase_note, level='WARN')

# Leases
expected_duration = leases['lease_months'].map({6: 24, 12: 48})
add_check('Logic', 'week_end - week_start ตรงกับระยะสัญญา',
          (leases['week_end'] - leases['week_start']).ne(expected_duration).sum())
add_check('Logic', 'is_renewal ตรงกับ tenant_segment=renewal',
          leases['is_renewal'].ne(leases['tenant_segment'].eq('renewal')).sum())
add_check('Domain', 'leases.rent เป็นบวก', leases['rent'].le(0).sum())

# Weekly market
add_check('Domain', 'occupancy อยู่ในช่วง 0–1', ~weekly['occupancy'].between(0, 1))
add_check('Domain', 'จำนวนใน weekly_market ไม่ติดลบ',
          (weekly[['n_searchers', 'n_listings_open', 'n_units_leased']] < 0).any(axis=1))
expected_occupancy = weekly['n_units_leased'] / weekly['project_id'].map(project_lookup['total_units'])
add_check('Logic', 'occupancy = n_units_leased / total_units',
          ~np.isclose(weekly['occupancy'], expected_occupancy))

# จำนวน units ต้องตรงกับ total_units ใน projects
actual_units = units.groupby('project_id').size().reindex(project_lookup.index, fill_value=0)
add_check('Cross-table', 'จำนวน units ต่อโครงการตรงกับ projects.total_units',
          actual_units.ne(project_lookup['total_units']).sum())

domain_logic_checks = pd.DataFrame(quality_checks)
display(domain_logic_checks[domain_logic_checks['area'].isin(['Domain', 'Logic'])])

,area,check,failures,status,note
22,Domain,units.floor อยู่ในช่วง 1–8,0,PASS,
23,Domain,units.size_sqm อยู่ในช่วงสมเหตุผล 20–50,0,PASS,
24,Domain,units.room_type อยู่ในหมวดที่กำหนด,0,PASS,
25,Domain,units.view อยู่ในหมวดที่กำหนด,0,PASS,
26,Domain,ค่าเช่าใน listings เป็นบวก,0,PASS,
27,Domain,weeks_on_market และ n_viewings ไม่ติดลบ,0,PASS,
28,Domain,season_listed อยู่ในหมวดที่กำหนด,0,PASS,
29,Logic,week_leased = week_listed + weeks_on_market เมื่อเช่าแล้ว,0,PASS,
30,Logic,asking_rent ไม่สูงกว่า first_asking_rent,285,WARN,พบ 285 รายการ; ทั้งหมดเป็น listing_id ที่ลงท้าย -B และเพิ่มสูงสุด 6.64% ควรยืนยันนิยาม first_ask...
31,Logic,week_end - week_start ตรงกับระยะสัญญา,0,PASS,


## 6. วันที่และลำดับสัปดาห์

สัปดาห์ที่ 0 เริ่มวันที่ 2023-01-03 ดังนั้นวันที่ในทั้งสามตารางต้องเท่ากับ `2023-01-03 + 7 × week`

In [7]:
base_date = pd.Timestamp('2023-01-03')
date_specs = [
    ('listings', 'date_listed', 'week_listed'),
    ('leases', 'date_start', 'week_start'),
    ('weekly_market', 'date', 'week'),
]
date_rows = []
for table_name, date_column, week_column in date_specs:
    frame = tables[table_name]
    parsed = pd.to_datetime(frame[date_column], errors='coerce')
    expected = base_date + pd.to_timedelta(frame[week_column] * 7, unit='D')
    invalid_dates = int(parsed.isna().sum())
    mismatches = int(parsed.ne(expected).sum())
    add_check('Date', f'{table_name}.{date_column} อ่านเป็นวันที่ได้', invalid_dates)
    add_check('Date', f'{table_name}: วันที่ตรงกับเลขสัปดาห์', mismatches)
    date_rows.append({
        'table': table_name,
        'invalid_dates': invalid_dates,
        'week_date_mismatches': mismatches,
        'min_date': parsed.min().date(),
        'max_date': parsed.max().date(),
    })

display(pd.DataFrame(date_rows))

,table,invalid_dates,week_date_mismatches,min_date,max_date
0,listings,0,0,2023-01-03,2026-06-23
1,leases,0,0,2023-01-03,2026-06-23
2,weekly_market,0,0,2023-01-03,2026-06-23


## 7. เปรียบเทียบ raw กับ processed

ตรวจว่าการทำความสะอาดรักษาจำนวนแถวไว้ และแก้ปัญหาหลักใน `listings` โดยไม่แก้ไฟล์ raw

In [8]:
comparison_rows = []
for table_name in TABLES:
    raw = raw_tables[table_name]
    clean = tables[table_name]
    comparison_rows.append({
        'table': table_name,
        'raw_rows': len(raw),
        'processed_rows': len(clean),
        'row_count_preserved': len(raw) == len(clean),
        'raw_missing_cells': int(raw.isna().sum().sum()),
        'processed_missing_cells': int(clean.isna().sum().sum()),
    })
    add_check('Raw→processed', f'{table_name}: จำนวนแถวไม่เปลี่ยน', len(raw) != len(clean))

display(pd.DataFrame(comparison_rows).style.format({
    'raw_rows': '{:,}', 'processed_rows': '{:,}',
    'raw_missing_cells': '{:,}', 'processed_missing_cells': '{:,}',
}))

raw_listings = raw_tables['listings']
clean_listings = tables['listings']
listing_cleanup = pd.DataFrame([
    {
        'issue': 'size_sqm นอกช่วง 20–50',
        'raw': int((~raw_listings['size_sqm'].between(20, 50)).sum()),
        'processed': int((~clean_listings['size_sqm'].between(20, 50)).sum()),
    },
    {
        'issue': 'asking_rent นอกช่วงตรวจสอบ 5,000–30,000',
        'raw': int((~raw_listings['asking_rent'].between(5_000, 30_000)).sum()),
        'processed': int((~clean_listings['asking_rent'].between(5_000, 30_000)).sum()),
    },
    {
        'issue': 'room_type มีช่องว่างหัว/ท้าย',
        'raw': int(raw_listings['room_type'].ne(raw_listings['room_type'].str.strip()).sum()),
        'processed': int(clean_listings['room_type'].ne(clean_listings['room_type'].str.strip()).sum()),
    },
    {
        'issue': 'view ไม่ใช่ตัวพิมพ์เล็ก',
        'raw': int(raw_listings['view'].ne(raw_listings['view'].str.lower()).sum()),
        'processed': int(clean_listings['view'].ne(clean_listings['view'].str.lower()).sum()),
    },
    {
        'issue': 'furnished หาย',
        'raw': int(raw_listings['furnished'].isna().sum()),
        'processed': int(clean_listings['furnished'].isna().sum()),
    },
    {
        'issue': 'date_listed ไม่ใช่รูปแบบ YYYY-MM-DD',
        'raw': int((~raw_listings['date_listed'].astype('string').str.fullmatch('[0-9]{4}-[0-9]{2}-[0-9]{2}')).sum()),
        'processed': int((~clean_listings['date_listed'].astype('string').str.fullmatch('[0-9]{4}-[0-9]{2}-[0-9]{2}')).sum()),
    },
])
display(listing_cleanup.style.format({'raw': '{:,}', 'processed': '{:,}'}))

,table,raw_rows,processed_rows,row_count_preserved,raw_missing_cells,processed_missing_cells
0,projects,19,19,True,0,0
1,units,"15,367","15,367",True,0,0
2,listings,"31,544","31,544",True,"6,835","1,698"
3,leases,"50,965","50,965",True,0,0
4,weekly_market,"2,288","2,288",True,3,3


,issue,raw,processed
0,size_sqm นอกช่วง 20–50,63,0
1,"asking_rent นอกช่วงตรวจสอบ 5,000–30,000",126,0
2,room_type มีช่องว่างหัว/ท้าย,"1,598",0
3,view ไม่ใช่ตัวพิมพ์เล็ก,"1,591",0
4,furnished หาย,"5,137",0
5,date_listed ไม่ใช่รูปแบบ YYYY-MM-DD,"31,544",0


## 8. สรุปผลตรวจทั้งหมด

In [9]:
check_results = pd.DataFrame(quality_checks)
status_order = pd.CategoricalDtype(['FAIL', 'WARN', 'PASS'], ordered=True)
check_results['status'] = check_results['status'].astype(status_order)
check_results = check_results.sort_values(['status', 'area', 'check']).reset_index(drop=True)
display(check_results)

n_fail = int(check_results['status'].eq('FAIL').sum()) + int((missing_checks['status'] == 'FAIL').sum())
n_warn = int(check_results['status'].eq('WARN').sum())
unleased_count = int((~listings['leased']).sum())
missing_median_count = int(weekly['median_asking_rent'].isna().sum())
absent_listing_projects = sorted(project_ids - set(listings['project_id']))

overall = 'PASS' if n_fail == 0 else 'FAIL'
display(Markdown(f'''
### ผลรวม: **{overall}**

- ข้อผิดพลาดร้ายแรง: **{n_fail}**
- ข้อควรติดตาม: **{n_warn}**
- ทั้ง 5 ตารางมี primary key ไม่ซ้ำ, foreign key ครบ และข้อมูลซ้ำข้ามตารางสอดคล้องกัน
- ค่าว่างในผลลัพธ์ listings จำนวน **{unleased_count:,} แถว** เกิดเฉพาะประกาศที่ยังไม่ถูกเช่า (censored records) จึงต้องเก็บไว้
- `median_asking_rent` ว่าง **{missing_median_count:,} แถว** และทุกแถวมี `n_listings_open=0` จึงสมเหตุผล
- โครงการที่ไม่มี listings/weekly market คือ project_id **{absent_listing_projects}** เพราะสร้างเสร็จหลังช่วงข้อมูล
- ข้อควรติดตาม: มี **{int(rent_increases.sum()):,}** ประกาศที่ `asking_rent > first_asking_rent`; ทั้งหมดเป็นประกาศรอบ `-B` ควรยืนยันนิยามก่อนสร้าง feature

**ข้อสรุป:** ข้อมูล processed พร้อมใช้สำหรับ EDA ขั้นถัดไป โดยห้ามทิ้ง 566 ประกาศที่ยังไม่ถูกเช่า และควรระวังความหมายของข้อมูลที่เกิดหลังวันลงประกาศเพื่อป้องกัน data leakage
'''))

,area,check,failures,status,note
0,Logic,asking_rent ไม่สูงกว่า first_asking_rent,285,WARN,พบ 285 รายการ; ทั้งหมดเป็น listing_id ที่ลงท้าย -B และเพิ่มสูงสุด 6.64% ควรยืนยันนิยาม first_ask...
1,Cross-table,leases.project_id ตรงกับ units.project_id,0,PASS,
2,Cross-table,listings.distance_to_campus_m ตรงกับ projects.distance_to_campus_m,0,PASS,
3,Cross-table,listings.facility_count ตรงกับ projects.facility_count,0,PASS,
4,Cross-table,listings.floor ตรงกับ units.floor,0,PASS,
5,Cross-table,listings.furnished ตรงกับ units.furnished,0,PASS,
6,Cross-table,listings.project_id ตรงกับ units.project_id,0,PASS,
7,Cross-table,listings.project_name ตรงกับ projects.project_name,0,PASS,
8,Cross-table,listings.room_type ตรงกับ units.room_type,0,PASS,
9,Cross-table,listings.size_sqm ตรงกับ units.size_sqm,0,PASS,



### ผลรวม: **PASS**

- ข้อผิดพลาดร้ายแรง: **0**
- ข้อควรติดตาม: **1**
- ทั้ง 5 ตารางมี primary key ไม่ซ้ำ, foreign key ครบ และข้อมูลซ้ำข้ามตารางสอดคล้องกัน
- ค่าว่างในผลลัพธ์ listings จำนวน **566 แถว** เกิดเฉพาะประกาศที่ยังไม่ถูกเช่า (censored records) จึงต้องเก็บไว้
- `median_asking_rent` ว่าง **3 แถว** และทุกแถวมี `n_listings_open=0` จึงสมเหตุผล
- โครงการที่ไม่มี listings/weekly market คือ project_id **[17]** เพราะสร้างเสร็จหลังช่วงข้อมูล
- ข้อควรติดตาม: มี **285** ประกาศที่ `asking_rent > first_asking_rent`; ทั้งหมดเป็นประกาศรอบ `-B` ควรยืนยันนิยามก่อนสร้าง feature

**ข้อสรุป:** ข้อมูล processed พร้อมใช้สำหรับ EDA ขั้นถัดไป โดยห้ามทิ้ง 566 ประกาศที่ยังไม่ถูกเช่า และควรระวังความหมายของข้อมูลที่เกิดหลังวันลงประกาศเพื่อป้องกัน data leakage
